# DRLB Smoke Demo: train_prior_lambda_init

Минимальный notebook для быстрой проверки DRLB на синтетических BAT-подобных данных.

Что показывает:
- обучение `DRLBBidder.fit(...)`
- рассчитанную `train_prior_lambda_init`
- training diagnostics (`lambda`, `dqn_loss`, `reward_net_loss`, `reward_signal`)
- один evaluation rollout через `simulate_campaign(...)`
- базовые метрики: spend, clicks, contacts, final balance

In [1]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from simulator.model.drlb_bidder import DRLBBidder
from simulator.simulation.modules import Campaign
from simulator.simulation.simulate import simulate_campaign

plt.style.use('ggplot')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def make_campaigns_df(num_campaigns: int = 8, hours: int = 24, budget: float = 120.0) -> pd.DataFrame:
    rows = []
    for campaign_id in range(1, num_campaigns + 1):
        rows.append(
            {
                'item_id': campaign_id,
                'campaign_id': campaign_id,
                'loc_id': 1,
                'region_id': 1,
                'logical_category': 'demo',
                'microcat_ext': 1,
                'campaign_start': 0,
                'campaign_end': hours * 3600,
                'auction_budget': float(budget),
            }
        )
    return pd.DataFrame(rows)


def make_stats_df(campaigns_df: pd.DataFrame, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    bins = [0, 4, 8, 12]
    rows = []

    for _, campaign in campaigns_df.iterrows():
        campaign_scale = 0.9 + 0.03 * int(campaign.campaign_id)
        total_hours = int((campaign.campaign_end - campaign.campaign_start) // 3600)

        for hour_idx in range(total_hours):
            period = int(campaign.campaign_start + hour_idx * 3600)
            hour_wave = 1.0 + 0.25 * np.sin(hour_idx / 3.0)
            ctr_pred = max(0.006, (0.012 + 0.002 * (hour_idx % 6)) * campaign_scale * hour_wave)
            cr_pred = max(0.03, 0.08 + 0.01 * np.cos(hour_idx / 4.0))

            for bin_idx, contact_price_bin in enumerate(bins):
                level = 1.0 + 0.55 * bin_idx
                noise = rng.uniform(0.95, 1.05)
                visibility = 55.0 * level * campaign_scale * noise
                clicks = visibility * ctr_pred * 0.14
                contacts = clicks * cr_pred
                spend = (0.35 + 0.28 * bin_idx) * campaign_scale * noise

                rows.append(
                    {
                        'campaign_id': int(campaign.campaign_id),
                        'period': period,
                        'contact_price_bin': int(contact_price_bin),
                        'CTRPredicts': float(ctr_pred),
                        'CRPredicts': float(cr_pred),
                        'AuctionWinBidSurplus': float(spend),
                        'AuctionVisibilitySurplus': float(visibility),
                        'AuctionClicksSurplus': float(clicks),
                        'AuctionContactsSurplus': float(contacts),
                    }
                )

    return pd.DataFrame(rows)


train_campaigns_df = make_campaigns_df(num_campaigns=8, hours=24, budget=120.0)
train_stats_df = make_stats_df(train_campaigns_df, seed=7)

train_campaigns_df.head(), train_stats_df.head()


(   item_id  campaign_id  loc_id  region_id logical_category  microcat_ext  campaign_start  campaign_end  \
 0        1            1       1          1             demo             1               0         86400   
 1        2            2       1          1             demo             1               0         86400   
 2        3            3       1          1             demo             1               0         86400   
 3        4            4       1          1             demo             1               0         86400   
 4        5            5       1          1             demo             1               0         86400   
 
    auction_budget  
 0           120.0  
 1           120.0  
 2           120.0  
 3           120.0  
 4           120.0  ,
    campaign_id  period  contact_price_bin  CTRPredicts  CRPredicts  AuctionWinBidSurplus  AuctionVisibilitySurplus  \
 0            1       0                  0     0.011160    0.090000              0.329572               

In [ ]:
params = {
    'exp_type': 'improved_drlb_eval',
    'objective': 'clicks',
    'eval_mode': False,
    'use_tqdm': False,
    'verbose': False,
    'debug_logs': False,
    'max_bid': 8.0,
    'min_bid': 0.0,
    'inference_lambda_init_mode': 'train_derived',
}

bidder = DRLBBidder(params)
bidder.fit(
    train_stats_df,
    campaigns_df=train_campaigns_df,
    objective='clicks',
)

diagnostics = bidder.get_training_diagnostics().copy()

print(f'train_prior_lambda_init = {bidder.train_prior_lambda_init:.6f}')
print(f'diagnostics rows = {len(diagnostics)}')
print(f'non-zero DQN losses = {(diagnostics["dqn_loss"] > 0).sum()}')
print(f'non-zero RewardNet losses = {(diagnostics["reward_net_loss"] > 0).sum()}')

diagnostics.head()


In [ ]:
summary_cols = ['lambda', 'dqn_loss', 'reward_net_loss', 'reward_signal']
display(diagnostics[summary_cols].describe())

fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=120)
axes = axes.ravel()

plot_specs = [
    ('lambda', 'Lambda'),
    ('dqn_loss', 'DQN Loss'),
    ('reward_net_loss', 'RewardNet Loss'),
    ('reward_signal', 'Reward Signal'),
]

for ax, (col, title) in zip(axes, plot_specs):
    ax.plot(diagnostics[col].to_numpy(), linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel('Training Step')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()


In [ ]:
# Save/load roundtrip to demonstrate that the trained prior is preserved for inference.
with tempfile.TemporaryDirectory() as tmpdir:
    model_path = Path(tmpdir) / 'drlb_smoke.pt'
    bidder.save_model(str(model_path))
    eval_bidder = DRLBBidder({
        **params,
        'model_path': str(model_path),
        'eval_mode': True,
        'use_tqdm': False,
    })

    eval_campaign_row = train_campaigns_df.iloc[0]
    eval_campaign = Campaign(
        item_id=int(eval_campaign_row.item_id),
        campaign_id=int(eval_campaign_row.campaign_id),
        loc_id=int(eval_campaign_row.loc_id),
        region_id=int(eval_campaign_row.region_id),
        logical_category=str(eval_campaign_row.logical_category),
        microcat_ext=int(eval_campaign_row.microcat_ext),
        campaign_start=int(eval_campaign_row.campaign_start),
        campaign_end=int(eval_campaign_row.campaign_end),
        initial_balance=float(eval_campaign_row.auction_budget),
        balance=float(eval_campaign_row.auction_budget),
        curr_time=int(eval_campaign_row.campaign_start),
        prev_time=int(eval_campaign_row.campaign_start),
        prev_balance=float(eval_campaign_row.auction_budget),
        prev_bid=0.0,
        prev_clicks=0.0,
        prev_contacts=0.0,
        desired_clicks=max(1, int(eval_campaign_row.auction_budget // 5.0)),
        desired_time=int((eval_campaign_row.campaign_end - eval_campaign_row.campaign_start) // 3600),
    )

    eval_stats_df = train_stats_df[train_stats_df.campaign_id == int(eval_campaign.campaign_id)].copy()
    sim_hist = simulate_campaign(
        campaign=eval_campaign,
        bidder=eval_bidder,
        stats_file=eval_stats_df,
        auction_mode='VCG',
    )

history_df = sim_hist.to_data_frame()
print(f'loaded train_prior_lambda_init = {eval_bidder.train_prior_lambda_init:.6f}')
print(f'rows in simulated history = {len(history_df)}')

history_df.head()


In [ ]:
rollout_summary = pd.Series(
    {
        'total_spend_history': history_df['spend_history'].sum(),
        'total_clicks_history': history_df['clicks_history'].sum(),
        'final_balance': history_df['balance'].iloc[-1],
        'final_clicks': history_df['clicks'].iloc[-1],
        'final_contacts': history_df['contacts'].iloc[-1],
        'avg_bid': history_df['bid'].mean(),
    }
)

display(rollout_summary.to_frame('value'))

fig, axes = plt.subplots(3, 1, figsize=(12, 9), dpi=120, sharex=True)
axes[0].plot(history_df['curr_timestamp'], history_df['balance'], marker='o')
axes[0].set_ylabel('Balance')
axes[0].set_title('Campaign Balance Over Time')

axes[1].plot(history_df['curr_timestamp'], history_df['clicks'], marker='o', color='tab:green')
axes[1].set_ylabel('Cumulative Clicks')
axes[1].set_title('Cumulative Clicks Over Time')

axes[2].plot(history_df['curr_timestamp'], history_df['bid'], marker='o', color='tab:blue')
axes[2].set_ylabel('Bid')
axes[2].set_xlabel('Timestamp')
axes[2].set_title('Bid Over Time')

plt.tight_layout()
plt.show()
